# Limpeza, Pré-processamento e Unificação dos Dados

**Disciplina:** IMD1152 – Coleta e Raspagem de Dados (IMD / UFRN)
**Projeto:** Mapeamento Multiplataforma de Gêneros e Subgêneros de Jogos Digitais
**Etapa:** Integrante 4 – Limpeza, pré-processamento e unificação

**Objetivo do notebook:** transformar os arquivos brutos gerados pelos coletores em **um único dataset padronizado**, pronto para a etapa de análise e visualização.

| Entrada (`data/raw/`) | Origem | Método de coleta |
| :--- | :--- | :--- |
| `steam_raw.json` | Steam / SteamSpy | API REST |
| `roblox_raw.json` | Roblox | API REST |
| `gog_raw.json` | GOG | Catálogo JSON |
| `itchio_raw.csv` | Itch.io | Web Scraping HTML |

**Saída:** `data/processed/games_unificado.csv`

**Etapas:**
1. Leitura dos dados brutos
2. Padronização do esquema (cada fonte → colunas comuns)
3. Limpeza: tipos, valores ausentes, ruídos e duplicados
4. Taxonomia unificada de gêneros (`processing/genre_mapping.json`)
5. Detecção de outliers (IQR)
6. Validação da qualidade e exportação

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config.settings import GENRE_MAPPING_PATH, PROCESSED_DIR, RAW_DIR

## 1. Leitura dos dados brutos

Cada coletor salvou os dados **fiéis à fonte**, por isso os formatos são heterogêneos: três arquivos JSON (APIs) e um CSV (scraping HTML).
Se algum arquivo ainda não existir, ele é apenas ignorado (com aviso), para que o notebook continue funcionando.

In [2]:
def carregar_json(nome_arquivo):
    caminho = RAW_DIR / nome_arquivo
    if not caminho.exists():
        print(f"Aviso: {nome_arquivo} não encontrado em data/raw/")
        return pd.DataFrame()
    with caminho.open(encoding="utf-8") as arquivo:
        return pd.DataFrame(json.load(arquivo))


def carregar_csv(nome_arquivo):
    caminho = RAW_DIR / nome_arquivo
    if not caminho.exists() or caminho.stat().st_size == 0:
        print(f"Aviso: {nome_arquivo} não encontrado (ou vazio) em data/raw/")
        return pd.DataFrame()
    return pd.read_csv(caminho, dtype=str, keep_default_na=False)


df_steam = carregar_json("steam_raw.json")
df_roblox = carregar_json("roblox_raw.json")
df_gog = carregar_json("gog_raw.json")
df_itch = carregar_csv("itchio_raw.csv")

**Problemas encontrados nos dados brutos**

| Problema | Exemplo |
| :--- | :--- |
| Nomes de colunas diferentes para a mesma informação | título: `name` (Steam/Roblox), `title` (GOG/Itch.io) |
| Estruturas aninhadas | Roblox `creator` é um dicionário; GOG `genres` e `developers` são listas |
| Números como texto | preço da Steam em centavos (`"1999"`); Itch.io `"$4.99"` ou `"Free"` |
| Moedas e escalas diferentes | Steam em USD, Roblox em Robux, GOG em moeda local; nota da GOG de 0 a 50 |
| Vocabulários de gênero diferentes | `FPS` (Steam), `Deathmatch Shooter` (Roblox), `Shooter` (GOG/Itch.io) |
| Registros repetidos | no Itch.io o mesmo jogo aparece em várias páginas de gênero/tag |
| Caracteres corrompidos | no Itch.io, `BrÃ¼tal Legend` em vez de `Brütal Legend` |

### 1.1. Correção de codificação de caracteres (Itch.io)

O Itch.io responde com `Content-Type: text/html` **sem informar o charset**. Nesse caso a biblioteca `requests` assume ISO-8859-1 (Latin-1), mas a página está em UTF-8. O resultado são textos corrompidos (*mojibake*): acentos, emojis e o símbolo `£` viram sequências como `Ã©` e `Â£`.

A correção desfaz a decodificação errada: o texto é convertido de volta para bytes em Latin-1 e decodificado como UTF-8. Se a conversão falhar, o texto já estava correto e é mantido.

In [3]:
def corrigir_codificacao(texto):
    try:
        return texto.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError, AttributeError):
        return texto


if not df_itch.empty:
    corrigido = df_itch.map(corrigir_codificacao)
    print("Valores corrigidos:", (corrigido != df_itch).sum().sum())
    df_itch = corrigido

Valores corrigidos: 471


## 2. Padronização do esquema

Cada fonte é convertida para o **mesmo conjunto de colunas** (unificação de esquema e alinhamento de nomenclatura).

| Coluna | Descrição | Steam | Roblox | GOG | Itch.io |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `plataforma` | origem do registro | steam | roblox | gog | itchio |
| `id_origem` | identificador na plataforma | `appid` | `id` (universo) | `id` | `url` |
| `titulo` | nome do jogo | `name` | `name` | `title` | `title` |
| `desenvolvedor` | desenvolvedor/criador | `developer` | `creator.name` | `developers` | `author` |
| `generos_originais` | rótulos de gênero da fonte (separados por `\|`) | `tags` (por votos) + `genre` | `genre_l2`, `genre_l1`, `genre` | `genres` | `genre` + tag da página |
| `preco` / `moeda` | preço e moeda originais | `price`/100, USD | `price`, Robux | `price.finalMoney` | `price` |
| `gratuito` | jogo gratuito? | preço = 0 | sem preço | preço = 0 | `"Free"` |
| `jogadores_ativos` | jogadores simultâneos no momento da coleta | `live_player_count` (ou `ccu`) | `playing` | — | — |
| `avaliacao_pct` | % de avaliações positivas (0–100) | `positive / (positive + negative)` | `upVotes / (upVotes + downVotes)` | `reviewsRating × 2` | — |
| `em_alta` | aparece em listas de tendência | `top100in2weeks` | sorts `top-trending` / `up-and-coming` | — | — |
| `url` | página do jogo | montada pelo `appid` | montada pelo `rootPlaceId` | `storeLink` | `url` |
| `data_coleta` | momento da coleta | `collected_at` | `collected_at` | `collected_at` | `collected_at` |

O símbolo — indica que a plataforma **não disponibiliza** a informação; esses campos ficam vazios (`NaN`).

In [4]:
COLUNAS = [
    "plataforma", "id_origem", "titulo", "desenvolvedor", "generos_originais",
    "preco", "moeda", "gratuito", "jogadores_ativos", "avaliacao_pct",
    "em_alta", "url", "data_coleta",
]


def juntar_rotulos(rotulos):
    vistos = []
    for rotulo in rotulos:
        if isinstance(rotulo, str) and rotulo.strip() and rotulo.strip() not in vistos:
            vistos.append(rotulo.strip())
    return "|".join(vistos)


def percentual_positivo(positivos, negativos):
    positivos = pd.to_numeric(positivos, errors="coerce")
    negativos = pd.to_numeric(negativos, errors="coerce")
    total = positivos + negativos
    return (positivos / total * 100).where(total > 0)


def eh_gratuito(preco):
    return pd.Series(preco == 0, dtype="boolean").mask(preco.isna())

In [5]:
def padronizar_steam(df):
    if df.empty:
        return pd.DataFrame(columns=COLUNAS)

    def generos_steam(linha):
        tags = linha["tags"] if isinstance(linha["tags"], dict) else {}
        tags_ordenadas = sorted(tags, key=tags.get, reverse=True)
        oficiais = str(linha["genre"]).split(",") if pd.notna(linha["genre"]) else []
        return juntar_rotulos(tags_ordenadas + oficiais)

    preco = pd.to_numeric(df["price"], errors="coerce") / 100
    ao_vivo = df["live_player_count"] if "live_player_count" in df else pd.Series(np.nan, index=df.index)
    jogadores = pd.to_numeric(ao_vivo, errors="coerce").fillna(
        pd.to_numeric(df["ccu"], errors="coerce")
    )

    return pd.DataFrame({
        "plataforma": "steam",
        "id_origem": df["appid"].astype(str),
        "titulo": df["name"],
        "desenvolvedor": df["developer"],
        "generos_originais": df.apply(generos_steam, axis=1),
        "preco": preco,
        "moeda": "USD",
        "gratuito": eh_gratuito(preco),
        "jogadores_ativos": jogadores,
        "avaliacao_pct": percentual_positivo(df["positive"], df["negative"]),
        "em_alta": df["ranking_sources"].apply(lambda fontes: "top100in2weeks" in fontes),
        "url": "https://store.steampowered.com/app/" + df["appid"].astype(str),
        "data_coleta": df["collected_at"],
    })


def padronizar_roblox(df):
    if df.empty:
        return pd.DataFrame(columns=COLUNAS)

    explore = df["explore_data"].apply(lambda d: d if isinstance(d, dict) else {})
    sorts = df["explore_sorts"].apply(lambda lista: {s["sort_id"] for s in lista})
    generos = df.apply(lambda l: juntar_rotulos([l["genre_l2"], l["genre_l1"], l["genre"]]), axis=1)

    return pd.DataFrame({
        "plataforma": "roblox",
        "id_origem": df["id"].astype(str),
        "titulo": df["name"],
        "desenvolvedor": df["creator"].str.get("name"),
        "generos_originais": generos,
        "preco": pd.to_numeric(df["price"], errors="coerce").fillna(0),
        "moeda": "Robux",
        "gratuito": df["price"].isna(),
        "jogadores_ativos": pd.to_numeric(df["playing"], errors="coerce"),
        "avaliacao_pct": percentual_positivo(
            explore.str.get("totalUpVotes"), explore.str.get("totalDownVotes")
        ),
        "em_alta": sorts.apply(lambda s: bool(s & {"top-trending", "up-and-coming"})),
        "url": "https://www.roblox.com/games/" + df["rootPlaceId"].astype(str),
        "data_coleta": df["collected_at"],
    })


def padronizar_gog(df):
    if df.empty:
        return pd.DataFrame(columns=COLUNAS)

    preco_final = df["price"].apply(lambda p: (p or {}).get("finalMoney") or {})
    preco = pd.to_numeric(preco_final.str.get("amount"), errors="coerce")

    return pd.DataFrame({
        "plataforma": "gog",
        "id_origem": df["id"].astype(str),
        "titulo": df["title"],
        "desenvolvedor": df["developers"].apply(lambda lista: juntar_rotulos(lista or [])),
        "generos_originais": df["genres"].apply(
            lambda lista: juntar_rotulos([g.get("name") for g in (lista or [])])
        ),
        "preco": preco,
        "moeda": preco_final.str.get("currency"),
        "gratuito": eh_gratuito(preco),
        "jogadores_ativos": np.nan,
        "avaliacao_pct": pd.to_numeric(df["reviewsRating"], errors="coerce") * 2,
        "em_alta": pd.NA,
        "url": df["storeLink"],
        "data_coleta": df["collected_at"],
        "tipo_produto": df["productType"],
    })


def padronizar_itch(df):
    if df.empty:
        return pd.DataFrame(columns=COLUNAS)

    rotulo_pagina = (
        df["source_path"].str.replace("/games/", "", regex=False)
        .str.replace(r"^(tag|genre)-", "", regex=True)
        .str.replace("-", " ", regex=False)
    )
    texto_preco = df["price"].str.strip()
    preco = pd.to_numeric(texto_preco.str.replace(r"[^\d.]", "", regex=True), errors="coerce")
    preco = preco.where(texto_preco.str.lower() != "free", 0.0)
    moeda = texto_preco.str.extract(r"^([^\d\s]+)", expand=False).replace(
        {"$": "USD", "€": "EUR", "£": "GBP", "R$": "BRL", "Free": np.nan}
    )

    return pd.DataFrame({
        "plataforma": "itchio",
        "id_origem": df["url"],
        "titulo": df["title"],
        "desenvolvedor": df["author"],
        "generos_originais": [juntar_rotulos([g, p]) for g, p in zip(df["genre"], rotulo_pagina)],
        "preco": preco,
        "moeda": moeda,
        "gratuito": eh_gratuito(preco),
        "jogadores_ativos": np.nan,
        "avaliacao_pct": np.nan,
        "em_alta": pd.NA,
        "url": df["url"],
        "data_coleta": df["collected_at"],
    })

In [6]:
padronizados = [
    padronizar_steam(df_steam),
    padronizar_roblox(df_roblox),
    padronizar_gog(df_gog),
    padronizar_itch(df_itch),
]

df = pd.concat(padronizados, ignore_index=True)

## 3. Limpeza de dados

### 3.1. Padronização de textos e representações de nulo

Na raspagem e nas APIs, a ausência de valor aparece de várias formas (`""`, `"N/A"`, `"null"`, `"?"`...). Primeiro removemos espaços extras dos textos e depois convertemos todas essas representações para o nulo oficial do pandas (`np.nan`).

In [7]:
colunas_texto = ["id_origem", "titulo", "desenvolvedor", "generos_originais", "moeda", "url"]
for coluna in colunas_texto:
    df[coluna] = df[coluna].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

substitutos = ["", "?", "N/A", "n/a", "NA", "null", "None", "nan", "-"]
df[colunas_texto] = df[colunas_texto].replace(substitutos, np.nan)

for coluna in ["preco", "jogadores_ativos", "avaliacao_pct"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
df["data_coleta"] = pd.to_datetime(df["data_coleta"], errors="coerce", utc=True)
for coluna in ["gratuito", "em_alta"]:
    df[coluna] = df[coluna].astype("boolean")

### 3.2. Ruídos e violações de regras de domínio

Valores **impossíveis** são ruído (erro), não informação real: preço ou jogadores negativos e percentual fora de 0–100. Esses valores viram `NaN`.

Também há uma inconsistência de domínio na GOG: o catálogo inclui **pacotes** (`productType = "pack"`), que são coleções de vários jogos e não um jogo individual. Eles são removidos para não contar o mesmo jogo duas vezes.

In [8]:
df.loc[df["preco"] < 0, "preco"] = np.nan
df.loc[df["jogadores_ativos"] < 0, "jogadores_ativos"] = np.nan
df.loc[~df["avaliacao_pct"].between(0, 100), "avaliacao_pct"] = np.nan

sem_avaliacao_gog = (df["plataforma"] == "gog") & (df["avaliacao_pct"] == 0)
print("GOG com nota 0 (sem avaliações):", sem_avaliacao_gog.sum())
df.loc[sem_avaliacao_gog, "avaliacao_pct"] = np.nan

pacotes = df["tipo_produto"] == "pack" if "tipo_produto" in df else pd.Series(False, index=df.index)
print("Pacotes GOG removidos:", pacotes.sum())
df = df[~pacotes].drop(columns="tipo_produto", errors="ignore").reset_index(drop=True)

GOG com nota 0 (sem avaliações): 1692
Pacotes GOG removidos: 1843


### 3.3. Valores ausentes

Resumo da quantidade e do percentual de valores ausentes por coluna:

In [9]:
resumo_nulos = pd.DataFrame({
    "Quantidade": df.isna().sum(),
    "Percentual (%)": (df.isna().mean() * 100).round(2),
})
resumo_nulos

,Quantidade,Percentual (%)
plataforma,0,0.00
id_origem,0,0.00
titulo,0,0.00
desenvolvedor,1,0.01
generos_originais,0,0.00
preco,305,2.47
moeda,4199,34.05
gratuito,305,2.47
jogadores_ativos,11494,93.21
avaliacao_pct,5654,45.85


**Decisão para cada caso** (remover, preencher ou manter e documentar):

| Coluna | Situação | Tratamento |
| :--- | :--- | :--- |
| `titulo`, `id_origem` | registro sem identificação não pode ser analisado | **remover** a linha |
| `desenvolvedor` | poucos ausentes; texto livre | **preencher** com `"Desconhecido"` (valor constante) |
| `generos_originais` | jogo sem rótulo de gênero na fonte | **manter**; recebe o macro-gênero `"Não classificado"` na etapa 4 |
| `jogadores_ativos` | ausente em 100% da GOG e do Itch.io (a fonte não fornece) | **manter `NaN`** – imputar inventaria jogadores; as análises de jogadores usam Steam e Roblox |
| `avaliacao_pct` | não existe no Itch.io e falta em jogos sem avaliações | **manter `NaN`** – preencher com média/mediana distorceria a distribuição |
| `jogadores_ativos` (Steam) | consulta em tempo real falhou para algum jogo | já **preenchido** na padronização com o `ccu` do SteamSpy (mesma medida, outra fonte) |
| `preco` (Roblox) | `price` nulo significa acesso gratuito, não dado faltante | já **preenchido com 0** na padronização |
| `preco` (GOG) | jogos "em breve" (`coming-soon`) ainda não têm preço | **manter `NaN`** – o preço ainda não existe |
| `moeda` | jogo gratuito no Itch.io não exibe moeda | **manter `NaN`** (não se aplica) |

In [10]:
antes = len(df)
df = df.dropna(subset=["titulo", "id_origem"])
print("Linhas removidas sem título/id:", antes - len(df))

print("Desenvolvedores preenchidos com 'Desconhecido':", df["desenvolvedor"].isna().sum())
df["desenvolvedor"] = df["desenvolvedor"].fillna("Desconhecido")

df = df.reset_index(drop=True)

Linhas removidas sem título/id: 0
Desenvolvedores preenchidos com 'Desconhecido': 1


### 3.4. Registros duplicados

Uma linha é duplicada quando a **mesma plataforma** traz o **mesmo identificador** mais de uma vez. Isso é esperado no Itch.io: o coletor percorre várias páginas de gênero e de tag, e um jogo popular aparece em várias delas.

Antes de remover as cópias, os rótulos de gênero de todas elas são **combinados**: se o jogo apareceu nas páginas *horror* e *pixel art*, ele fica com os dois rótulos.

Jogos com o mesmo título em plataformas diferentes (ex.: um jogo vendido na Steam e na GOG) **não** são duplicados: cada linha representa o jogo em uma plataforma.

In [11]:
chave = ["plataforma", "id_origem"]

generos_combinados = df.groupby(chave)["generos_originais"].agg(
    lambda serie: juntar_rotulos("|".join(serie.dropna()).split("|"))
)

antes = len(df)
df = df.drop_duplicates(subset=chave, keep="first").set_index(chave)
df["generos_originais"] = generos_combinados.replace("", np.nan)
df = df.reset_index()

print(f"Duplicados removidos: {antes - len(df)}  |  linhas restantes: {len(df)}")

Duplicados removidos: 1415  |  linhas restantes: 10916


## 4. Taxonomia unificada de gêneros

Cada plataforma usa um vocabulário próprio. Exemplos do mesmo tipo de jogo:

| Macro-gênero | Steam | Roblox | GOG | Itch.io |
| :--- | :--- | :--- | :--- | :--- |
| Tiro | `FPS`, `Hero Shooter` | `Deathmatch Shooter`, `PvE Shooter` | `Shooter` | `Shooter` |
| Plataforma | `Platformer`, `Metroidvania` | `Classic Obby`, `Tower Obby` | `Platformer` | `Platformer` |
| RPG | `Souls-like`, `JRPG` | `Action RPG`, `Turn-based RPG` | `Role-playing` | `Role Playing`, `jrpg` |

O dicionário `processing/genre_mapping.json` agrupa esses rótulos em **macro-gêneros comparáveis**. Formato: `{"Macro-gênero": ["rótulo 1", "rótulo 2", ...]}`.

**Regra de classificação:** os rótulos de cada jogo estão ordenados do mais importante para o menos importante (tags mais votadas na Steam, subgênero antes do gênero na Roblox). O **primeiro rótulo que existe no dicionário** define o `macro_genero`, e esse rótulo original é guardado em `subgenero`. Rótulos que não descrevem gênero (ex.: `Free to Play`, `Multiplayer`, `Indie`) não estão no dicionário e são ignorados.

In [12]:
def normalizar_rotulo(texto):
    return " ".join(str(texto).lower().replace("-", " ").replace("_", " ").split())


with open(GENRE_MAPPING_PATH, encoding="utf-8") as arquivo:
    genre_mapping = json.load(arquivo)

rotulo_para_macro = {
    normalizar_rotulo(rotulo): macro
    for macro, rotulos in genre_mapping.items()
    for rotulo in rotulos
}


def classificar_genero(generos_originais):
    if pd.isna(generos_originais):
        return "Não classificado", np.nan
    for rotulo in generos_originais.split("|"):
        macro = rotulo_para_macro.get(normalizar_rotulo(rotulo))
        if macro:
            return macro, rotulo
    return "Outros", np.nan


classificacao = df["generos_originais"].apply(classificar_genero)
df["macro_genero"] = classificacao.str[0]
df["subgenero"] = classificacao.str[1]

## 5. Detecção de outliers (IQR)

A variável `jogadores_ativos` tem escalas muito diferentes entre plataformas, então um valor só é atípico **em relação à própria plataforma** (outlier contextual). Por isso o método do **intervalo interquartil** é aplicado separadamente para Steam e Roblox:

- Limite inferior: $Q_1 - 1{,}5 \times IQR$
- Limite superior: $Q_3 + 1{,}5 \times IQR$

Aqui os outliers são **informação real** (os jogos mais populares do mercado), e não erros de medição. Por isso eles **não são removidos**: são apenas marcados na coluna `outlier_jogadores`, para que a análise possa decidir se os inclui ou não.

In [13]:
def limites_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


df["outlier_jogadores"] = pd.Series(pd.NA, index=df.index, dtype="boolean")

for plataforma, grupo in df.groupby("plataforma"):
    valores = grupo["jogadores_ativos"].dropna()
    if valores.empty:
        continue
    inferior, superior = limites_iqr(valores)
    mascara = (valores < inferior) | (valores > superior)
    df.loc[valores.index, "outlier_jogadores"] = mascara
    print(f"{plataforma}: {mascara.sum()} outliers de {len(valores)} jogos (limite superior: {superior:.0f})")

roblox: 86 outliers de 737 jogos (limite superior: 19214)
steam: 10 outliers de 100 jogos (limite superior: 65530)


## 6. Validação da qualidade

Conferências finais (consistência): não pode haver chaves repetidas nem jogos sem título ou sem macro-gênero.

In [14]:
assert not df.duplicated(subset=["plataforma", "id_origem"]).any(), "Ainda existem duplicados!"
assert df["titulo"].notna().all() and df["macro_genero"].notna().all()
print("Sem duplicados e sem títulos/macro-gêneros ausentes.")

Sem duplicados e sem títulos/macro-gêneros ausentes.


## 7. Exportação

O dataset final é salvo em `data/processed/games_unificado.csv` (UTF-8, separador vírgula).

**Dicionário de dados do arquivo final**

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `plataforma` | texto | `steam`, `roblox`, `gog` ou `itchio` |
| `id_origem` | texto | identificador do jogo na plataforma |
| `titulo` | texto | nome do jogo |
| `desenvolvedor` | texto | desenvolvedor/criador (`Desconhecido` quando ausente) |
| `macro_genero` | texto | gênero unificado entre plataformas (`genre_mapping.json`) |
| `subgenero` | texto | rótulo original que definiu o macro-gênero (ex.: `FPS`, `Classic Obby`) |
| `generos_originais` | texto | todos os rótulos de gênero da fonte, separados por `\|` |
| `preco` | número | preço na moeda original (vazio se desconhecido) |
| `moeda` | texto | `USD`, `Robux`, moeda local da GOG etc. |
| `gratuito` | booleano | jogo gratuito / acesso livre |
| `jogadores_ativos` | número | jogadores simultâneos no momento da coleta (Steam e Roblox) |
| `outlier_jogadores` | booleano | `True` se `jogadores_ativos` é outlier (IQR) dentro da plataforma |
| `avaliacao_pct` | número | % de avaliações positivas (0–100) |
| `em_alta` | booleano | jogo presente em lista de tendência (Steam e Roblox) |
| `url` | texto | página do jogo |
| `data_coleta` | data/hora | momento da coleta (UTC) |

In [15]:
COLUNAS_FINAIS = [
    "plataforma", "id_origem", "titulo", "desenvolvedor", "macro_genero", "subgenero",
    "generos_originais", "preco", "moeda", "gratuito", "jogadores_ativos",
    "outlier_jogadores", "avaliacao_pct", "em_alta", "url", "data_coleta",
]

df_final = df[COLUNAS_FINAIS].sort_values(["plataforma", "titulo"]).reset_index(drop=True)
df_final["avaliacao_pct"] = df_final["avaliacao_pct"].round(2)
df_final["jogadores_ativos"] = df_final["jogadores_ativos"].astype("Int64")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
caminho_saida = PROCESSED_DIR / "games_unificado.csv"
df_final.to_csv(caminho_saida, index=False, encoding="utf-8")

print(f"Arquivo salvo em: {caminho_saida.relative_to(ROOT)}")
print("Dimensões:", df_final.shape)

Arquivo salvo em: data\processed\games_unificado.csv
Dimensões: (10916, 16)
